###**Interpolação de Nulos, Tratamento de Outliers e Salvamento em data/processed**
1. Carregar os dados consolidados

In [6]:
# MOntando drive e carregando dados brutos

from google.colab import drive
import numpy as np
import pandas as pd

drive.mount("/content/drive")
%cd /content/drive/MyDrive/cafeicultura-varginha-analytics

# Carrega os dados brutos e garante o tipo datetime
df = pd.read_csv("data/raw/varginha_2012_2024_consolidado.csv")
df["data_hora"] = pd.to_datetime(df["data_hora"])

2. Tratar outliers e aplicar interpolação linear

In [13]:
# 1. Identifica apenas as colunas contínuas existentes no dataset
colunas_alvo = [
    'temp_ar_c',
    'temp_max_c',
    'temp_min_c',
    'umidade_rel_pct',
    'vento_velocidade_ms',
]
col_continuas = [c for c in colunas_alvo if c in df.columns]

# Outliers
if 'temp_ar_c' in df.columns:
    df.loc[(df['temp_ar_c'] < -5) | (df['temp_ar_c'] > 45), 'temp_ar_c'] = np.nan
if 'umidade_rel_pct' in df.columns:
    df.loc[
        (df['umidade_rel_pct'] < 0) | (df['umidade_rel_pct'] > 100),
        'umidade_rel_pct',
    ] = np.nan

# 2. Interpolação linear segura (máximo 2 horas consecutivas)
df[col_continuas] = df[col_continuas].interpolate(method='linear', limit=2)

# 3. Preenchimento de lacunas residuais maiores
df[col_continuas] = df[col_continuas].ffill().bfill()

# 4. Salva o dataset limpo em data/processed/
caminho_processado = 'data/processed/varginha_2012_2024_tratado.csv'
df.to_csv(caminho_processado, index=False)

print(f'✔ Dataset limpo salvo em: {caminho_processado}')
print(f'Total de nulos remanescentes: {df[col_continuas].isna().sum().sum()}')

print(df.columns)

,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (KJ/m²),RADIACAO GLOBAL (Kj/m²),TEMPERATURA DO PONTO DE ORVALHO (°C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),...,umidade_rel_pct,hora_limpa,data_hora,ano,mes,dia,hora_num,dia_do_ano,horas_estresse_termico,horas_risco_geada
0,0.6,904.5,904.5,904.0,NaN,NaN,18.9,19.1,18.9,96.0,...,96.0,00:00,2012-01-01 00:00:00,2012,1,1,0,1,0,0
1,2.2,904.7,904.7,904.5,NaN,NaN,18.7,18.9,18.7,97.0,...,97.0,01:00,2012-01-01 01:00:00,2012,1,1,1,1,0,0
2,0.6,904.2,904.7,904.1,NaN,NaN,19.0,19.0,18.7,97.0,...,97.0,02:00,2012-01-01 02:00:00,2012,1,1,2,1,0,0
3,2.6,903.5,904.2,903.5,NaN,NaN,18.7,19.0,18.7,97.0,...,97.0,03:00,2012-01-01 03:00:00,2012,1,1,3,1,0,0
4,3.2,902.4,903.5,902.4,NaN,NaN,19.0,19.0,18.7,98.0,...,98.0,04:00,2012-01-01 04:00:00,2012,1,1,4,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113971,0.0,905.0,906.0,905.0,NaN,2564.2,15.8,17.8,14.1,50.0,...,44.0,19:00,2024-12-31 19:00:00,2024,12,31,19,366,0,0
113972,0.0,905.2,905.3,904.9,NaN,1063.2,14.5,15.9,14.1,47.0,...,44.0,20:00,2024-12-31 20:00:00,2024,12,31,20,366,0,0
113973,0.0,905.8,905.8,905.1,NaN,273.9,17.7,17.7,14.1,68.0,...,68.0,21:00,2024-12-31 21:00:00,2024,12,31,21,366,0,0
113974,1.2,906.8,907.2,905.6,NaN,3.9,15.2,17.8,12.5,89.0,...,89.0,22:00,2024-12-31 22:00:00,2024,12,31,22,366,0,0


###**Cálculo de dados bioclimáticos**
1. GDA (Graus-Dia Acumulados)
2. Amplitude térmica diária
3. Horas de estresse térmico
4. Horas de risco de geada



In [26]:
# 1. Indicadores horários
df["horas_estresse_termico"] = (df["temp_ar_c"] >= 30).astype(int)
df["horas_risco_geada"] = (df["temp_ar_c"] <= 4).astype(int)

# 2. Agregação diária (resumo por data)
df_diario = (
    df.groupby("data")
    .agg(
        temp_media=("temp_ar_c", "mean"),
        temp_max=("temp_ar_c", "max"),
        temp_min=("temp_ar_c", "min"),
        umidade_media=("umidade_rel_pct", "mean"),
        precipitacao_total=("PRECIPITAÇÃO TOTAL, HORÁRIO (mm)", "sum"),
        horas_estresse_termico=("horas_estresse_termico", "sum"),
        horas_risco_geada=("horas_risco_geada", "sum"),
    )
    .reset_index()
)

# 3. Métricas diárias calculadas
df_diario["amplitude_termica"] = (
    df_diario["temp_max"] - df_diario["temp_min"]
)

#-----------------------------------------------------------------------------------------------
# Calcula Graus-Dia Acumulados (GDA) pelo Método de Ometto (1981).

def calcular_gda_ometto(df, tb=10.0, tot=26.0, TB=32.0):
    """
    Parâmetros:
    tb  : Temperatura Base Inferior (10°C para C. arabica)
    tot : Temperatura Ótima (26°C)
    TB  : Temperatura Base Superior (32°C)
    """
    tmax = df["temp_max"]
    tmin = df["temp_min"]
    A = tmax - tmin

    # Evita divisão por zero se Tmax == Tmin
    A = np.where(A == 0, 0.0001, A)

    # -------------------------------------------------------------
    # Fórmulas de Ometto para cada caso:
    # -------------------------------------------------------------

    # Caso 1: TB > Tot >= Tmax >= Tmin >= Tb
    gda_1 = (tmax + tmin) / 2.0 - tb

    # Caso 2: TB > Tot >= Tmax > Tb > Tmin
    gda_2 = ((tmax - tb) ** 2) / (2.0 * A)

    # Caso 3: TB >= Tmax > Tot > Tmin >= Tb
    gda_3 = ((tmax - tb) ** 2 - (tmax - tot) ** 2) / (2.0 * A)

    # Caso 4: TB >= Tmax > Tot > Tb > Tmin
    gda_4 = ((tmax - tb) ** 2 - (tmax - tot) ** 2) / (2.0 * A)

    # Caso 5: Tmax > TB > Tot > Tmin >= Tb
    W = tmax - TB
    gda_5 = ((tmax - tb) ** 2 - (tmax - tot) ** 2 - W**2) / (2.0 * A)

    # -------------------------------------------------------------
    # Condições de Seleção:
    # -------------------------------------------------------------

    condicoes = [
        # Fora dos limites (dia frio demais, sem acúmulo)
        (tmax <= tb),
        # Caso 1
        (tmax <= tot) & (tmin >= tb),
        # Caso 2
        (tmax <= tot) & (tmin < tb),
        # Caso 3
        (tmax > tot) & (tmax <= TB) & (tmin >= tb),
        # Caso 4
        (tmax > tot) & (tmax <= TB) & (tmin < tb),
        # Caso 5
        (tmax > TB) & (tmin >= tb),
    ]

    escolhas = [0.0, gda_1, gda_2, gda_3, gda_4, gda_5]

    # numpy.select avalia as condições em ordem e aplica a fórmula adequada
    return np.select(condicoes, escolhas, default=0.0)


# Aplicação direta no DataFrame:
df_diario["graus_dia_ometto"] = calcular_gda_ometto(
    df_diario, tb=10, tot=26, TB=32
)
#------------------------------------------------------------------------------------------------------
# Datetime e ordenação
df_diario["data"] = pd.to_datetime(df_diario["data"], format="mixed")
df_diario = df_diario.sort_values("data").reset_index(drop=True)

df_diario["ano"] = df_diario["data"].dt.year
df_diario["mes"] = df_diario["data"].dt.month

# 4. Salva dataset diário com features bioclimáticas
caminho_diario = "data/processed/varginha_diario_features.csv"
df_diario.to_csv(caminho_diario, index=False)

print(f"✔ Dataset diário gerado e salvo em: {caminho_diario}")
df_diario.head(3)

✔ Dataset diário gerado e salvo em: data/processed/varginha_diario_features.csv


,data,temp_media,temp_max,temp_min,umidade_media,precipitacao_total,horas_estresse_termico,horas_risco_geada,amplitude_termica,graus_dia_ometto,ano,mes
0,2012-01-01,20.016667,22.1,19.1,95.833333,27.2,0,0,3.0,10.600000,2012,1
1,2012-01-02,19.829167,22.9,17.9,92.916667,22.6,0,0,5.0,10.400000,2012,1
2,2012-01-03,21.462500,27.6,17.7,79.666667,0.4,0,0,9.9,15.515152,2012,1


In [28]:
!git add 02_limpeza_tratamento.ipynb
!git commit -m "feat: calculo de indicadores bioclimaticos e agregacao diaria"
!git push origin main

fatal: pathspec '02_limpeza_tratamento.ipynb' did not match any files


In [31]:
!ls *.ipynb
!git add 02_limpeza_tratamento.ipynb
!git commit -m "feat: adiciona notebook 02 com limpeza e features"
!git push origin main

01_coleta_dados.ipynb  02_limpeza_tratamento.ipynb
